# Deterministic sampling

Relationship discovery is bounded by design: broad screening over
everything, expensive statistics over a sample. This chapter describes the
samplers — how rows are chosen, which passes use which sampler, and what
the choice costs. The contract is the same everywhere: a sample is
deterministic, disclosed in coverage, and never silently substituted for a
full scan where exactness matters (keys, cards, duplicates, sortedness,
identity verification).

## The stratified sampler

`sample_rows(n, cap)` picks `cap` rows from `n` by splitting the row range
into `cap` equal strata and choosing one row from each stratum. The choice
within a stratum is a hashed offset of the stratum index — a fixed
arithmetic mix, so the same input always yields the same rows — and the
result is sorted. Three properties follow:

1. **Every part of the table is represented.** Row 0 and row n−1 strata
   always contribute; the head and tail of a sorted extract cannot fall
   outside the sample.
2. **No periodic aliasing.** A plain stride (every k-th row) resonates with
   periodic data — weekly cycles, sharded exports — and can systematically
   sample one weekday. A hashed offset per stratum does not.
3. **Reproducibility.** Same table, same rows, no stored state, no
   wall-clock input. Sampling is not a source of nondeterminism.

When the table fits under the cap, the "sample" is simply every row.

The frame used below makes the different sampling regimes visible in one
coverage block: 300,000 rows, an `id`, ten random columns, a target, and
one planted lookup pair (`region` → `region_name`) so there is something
for the dependency pass to verify.

In [ ]:
import numpy as np
import polars as pl

import onceover

rng = np.random.default_rng(3)
n = 300_000
region = rng.integers(0, 5, n)
big = pl.DataFrame(
    {"id": np.arange(n, dtype=np.int64)}
    | {"region": region}
    | {"region_name": [f"r{r}" for r in region]}
    | {f"x{i:02}": rng.random(n) for i in range(10)}
    | {"y": rng.random(n)}
)

report = onceover.profile(big)
for name in ["encoding", "predictability_screen", "functional_dependencies", "dependence_screen"]:
    print(f"{name:24s}", report.coverage[name]["detail"])
    print()

The coverage detail pairs every sample with its population — "20000 rows
(20000 of 300000)" — so a reader always knows which claims were measured
where. On this frame the encoded matrix and the dependence statistics run
over 20,000 stratified rows; the FD screen looks at 26 directed pairs
over 2,048 rows, then verifies its two discovered candidates over a
200,000-row sample; the column cards and keys never sample at all.

## Which pass uses which rows

| Pass | Rows used (`standard` / `deep`) | Mechanism |
|---|---|---|
| Column cards, duplicates, single keys, sortedness, dates | all rows | exact |
| FD broad screen | 2,048 | stratified sample of hashed values |
| FD verification | ≤ 200,000 / 500,000 | stratified sample, canonical value bytes |
| Composite-key screen / verification | 2,048 screen; ≤ 16 / 24 verified pairs, full data | hash screen, exact count |
| Encoded matrix | ≤ 20,000 / 50,000 | stratified |
| Predictability screen | ≤ 5,000 / 10,000 fitting rows | seeded shuffle, 75/25 holdout per target |
| Decision tables | encoded rows | 5-fold cross-validation |
| PRISM rules | encoded rows | bitset masks over the encoded sample |
| Rule evaluation | 1/5 of sampled rows | row-id hash partition |
| Association mining | 4/5 train, 1/5 holdout | row-id hash partition |
| Dependence statistics | ≤ 20,000 / 50,000 | stratified, pair-complete |
| Identity fitting | ≤ 20,000 | stratified |
| Identity residual check | all pair-complete rows | exact |
| Regimes | ≤ 5,000 / 10,000 | 1/5 row-id holdout |

Two partition mechanisms recur. The predictability screen shuffles
positions with a seeded generator and holds out 25% per target — the
holdout never trained that model. The rule evaluator, the association
miner, and the regimes pass use a stable hash of the *original row id* to
split rows; because the same hash defines evaluation rows across passes, a
rule's evaluation partition is reproducible — but the mining passes may
already have seen those rows during discovery, which is why their
validation is labeled `diagnostic_partition` (or `diagnostic_holdout`) and
marked `used_for_selection`.

## What samples can miss

A 20,000-row sample cannot see a pattern that lives in 0.01% of rows.
Per-feature shortlisting cannot keep a signal that only exists as an
interaction of two weak features. A capped FD lattice stops at its
per-target determinant shortlist. None of these limits is silent: the
sample sizes, caps, and shortlists are in coverage on every run, and the
overrides (`sample_rows`, `max_predictors`, `max_pairs`, budget `deep`)
rescale exactly the sampled work.

A finding from a sample applies to the disclosed rows. If no finding
appears, increase the budget or test the specific hypothesis directly;
absence at a disclosed search bound does not prove independence.